# Lecture 13: Compatible Triples

**Source span.** Printed pages 72-75; physical PDF pages 86-89 in `Lectures on Symplectic Geometry.pdf`. I inspected the local PDF text for this span before revising the notebook.

**Lecture goal.** Package symplectic forms, almost complex structures, and metrics as a mutually determining compatible triple, then use that package to understand contractible choices, deformation-equivalent forms, and almost-complex submanifolds.

The lecture makes compatibility less mysterious: any two of `omega`, `J`, and `g` can reconstruct the third, provided the positivity and orthogonality conditions are present. The same viewpoint explains why compatible choices are contractible, why two forms compatible with one `J` are joined by a symplectic path, and why `J`-invariant submanifolds inherit symplectic forms.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np


def find_book_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "Lectures on Symplectic Geometry.pdf").exists():
            return candidate
    raise RuntimeError("Could not locate the Lectures-on-Symplectic-Geometry course root.")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, read_json, save_json, save_matplotlib
from utils.symplectic import standard_omega

ARTIFACT_TOPIC = "lecture-13"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / ARTIFACT_TOPIC
(ARTIFACT_ROOT / "figures").mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / "checks").mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
print(f"Artifact topic: {ARTIFACT_TOPIC}")

## Translation Guide And Library Routing

| Source idea | Computational object | Check |
| --- | --- | --- |
| compatible triple | matrices `Omega`, `J`, `G` | `G=Omega @ J`, `Omega=G @ (-J)`, and `J=Omega^{-1} @ G` with consistent conventions |
| Hermitian metric | real metric plus two-form packaged by `J` | `J.T @ G @ J = G` and `J.T @ Omega @ J = Omega` |
| contractible choice | model spaces: symmetric graph matrices and positive metrics | linear/convex homotopies contract to a base point |
| orthogonality | `J` preserves `G` | `||J.T G J - G||=0` |
| deformation equivalent | convex path of two `J`-compatible symplectic forms | induced metrics stay positive |
| almost complex submanifold | tangent subspace invariant under `J` | restricted two-form is nondegenerate |
| counterexample warning | symplectic path from `Omega` to `-Omega` in `R4` | path stays nondegenerate but no single `J` can be compatible with both endpoints |

**Library routing.** `numpy` is enough for the matrix reconstruction, Hermitian/orthogonality checks, deformation residuals, and subspace restrictions. `matplotlib` renders the reconstruction diagram, contraction model, and deformation/submanifold ledgers. `networkx` keeps the compatibility table as a proof dependency graph rather than a decorative slogan.

## Visualization Storyboard

1. **`omega-J-g` reconstruction diagram.** A graph and matrix heatmaps show how each pair determines the third and which differential question remains: flat metric, closed form, or integrable `J`.
2. **Contractible choice model.** Homework 9 identifies compatible `J` choices with a Lagrangian graph space and a positive-metric space; symmetric matrices contract linearly and positive metrics contract convexly.
3. **Consequences ledger.** A deformation-equivalence plot tracks positivity along a convex path, a counterexample path shows why the converse fails, and a subspace bar chart distinguishes `J`-invariant symplectic subspaces from non-invariant isotropic ones.

In [ ]:
# Triple reconstruction in the standard model.
Omega = standard_omega(2)
J = -Omega
Gmat = Omega @ J
Omega_from_gJ = Gmat @ (-J)
J_from_omega_g = np.linalg.solve(Omega, Gmat)
metric_reconstruction_residual = float(np.linalg.norm(Gmat - Omega @ J))
omega_reconstruction_residual = float(np.linalg.norm(Omega_from_gJ - Omega))
J_reconstruction_residual = float(np.linalg.norm(J_from_omega_g - J))
orthogonality_residual = float(np.linalg.norm(J.T @ Gmat @ J - Gmat))
hermitian_symplectic_residual = float(np.linalg.norm(J.T @ Omega @ J - Omega))
assert metric_reconstruction_residual < 1e-12
assert omega_reconstruction_residual < 1e-12
assert J_reconstruction_residual < 1e-12
assert orthogonality_residual < 1e-12
assert hermitian_symplectic_residual < 1e-12

Ggraph = nx.DiGraph()
Ggraph.add_edges_from([
    ("omega + J", "g(u,v)=omega(u,Jv)"),
    ("g + J", "omega(u,v)=g(Ju,v)"),
    ("omega + g", "polar decomposition gives J"),
    ("g(u,v)=omega(u,Jv)", "ask: is g flat?"),
    ("omega(u,v)=g(Ju,v)", "ask: is omega closed?"),
    ("polar decomposition gives J", "ask: is J integrable?"),
])
pos = {
    "omega + J": (0, 1.2),
    "g + J": (0, 0),
    "omega + g": (0, -1.2),
    "g(u,v)=omega(u,Jv)": (2.6, 1.2),
    "omega(u,v)=g(Ju,v)": (2.6, 0),
    "polar decomposition gives J": (2.6, -1.2),
    "ask: is g flat?": (5.1, 1.2),
    "ask: is omega closed?": (5.1, 0),
    "ask: is J integrable?": (5.1, -1.2),
}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.9), gridspec_kw={"width_ratios": [1.35, 1]})
nx.draw_networkx_edges(Ggraph, pos, ax=axes[0], arrows=True, arrowstyle="-|>", arrowsize=14, edge_color="0.35")
nx.draw_networkx_nodes(Ggraph, pos, ax=axes[0], node_color="#b3cde3", node_size=1900, edgecolors="0.25", linewidths=0.8)
nx.draw_networkx_labels(Ggraph, pos, ax=axes[0], font_size=8)
axes[0].set_axis_off()
axes[0].set_title("compatible triple reconstruction table")
ledger = [
    f"G - Omega J: {metric_reconstruction_residual:.1e}",
    f"Omega from (g,J): {omega_reconstruction_residual:.1e}",
    f"J from (omega,g): {J_reconstruction_residual:.1e}",
    f"J orthogonal for g: {orthogonality_residual:.1e}",
    f"Hermitian/symplectic residual: {hermitian_symplectic_residual:.1e}",
]
axes[1].axis("off")
axes[1].text(0.03, 0.95, "\n".join(ledger), va="top", fontsize=11, bbox={"boxstyle": "round,pad=0.45", "fc": "white", "ec": "0.55"})
fig.suptitle("The compatible triple is a reconstruction machine, not three unrelated objects")
triple_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "omega-J-g-reconstruction-diagram.png")
plt.close(fig)
display_artifact(triple_path, width=760)
print({"orthogonality_residual": orthogonality_residual})

In [ ]:
# Contractible model from Homework 9: symmetric Lagrangian graphs and positive metrics.
S0 = np.array([[0.8, -0.35], [-0.35, 0.25]])
P0 = np.array([[1.7, 0.35], [0.35, 0.95]])
assert np.allclose(S0, S0.T)
assert np.min(np.linalg.eigvalsh(P0)) > 0
homotopy_t = np.linspace(0, 1, 31)
S_norms = []
P_min_eigs = []
for t in homotopy_t:
    S_t = (1 - t) * S0
    P_t = (1 - t) * P0 + t * np.eye(2)
    S_norms.append(float(np.linalg.norm(S_t)))
    P_min_eigs.append(float(np.min(np.linalg.eigvalsh(P_t))))
assert S_norms[-1] < 1e-12
assert min(P_min_eigs) > 0

u = np.linspace(-1, 1, 11)
U, V = np.meshgrid(u, u)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.1))
for t, color in [(0.0, "#d95f02"), (0.5, "#2c7fb8"), (1.0, "#1b9e77")]:
    S_t = (1 - t) * S0
    X = U
    Y = V
    PX = S_t[0, 0] * U + S_t[0, 1] * V
    PY = S_t[1, 0] * U + S_t[1, 1] * V
    axes[0].quiver(X, Y, PX, PY, color=color, alpha=0.75, label=f"t={t:.1f}")
axes[0].set_aspect("equal", adjustable="box")
axes[0].set_title("Lagrangian graph matrix contracts to 0")
axes[0].set_xlabel("base e1")
axes[0].set_ylabel("base e2")
axes[0].legend(fontsize=8)
axes[1].plot(homotopy_t, S_norms, color="#d95f02", lw=2)
axes[1].set_title("symmetric graph norm")
axes[1].set_xlabel("homotopy t")
axes[1].set_ylabel("||S_t||")
axes[2].plot(homotopy_t, P_min_eigs, color="#1b9e77", lw=2)
axes[2].set_title("positive metrics stay positive")
axes[2].set_xlabel("homotopy t")
axes[2].set_ylabel("min eigenvalue of P_t")
fig.suptitle("Contractible choice: symmetric graph space and positive metrics both retract")
contractible_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "contractible-compatible-choice-model.png")
plt.close(fig)
display_artifact(contractible_path, width=760)
print({"S_final_norm": S_norms[-1], "P_min_eig_floor": min(P_min_eigs)})

In [ ]:
# Consequences: compatible forms deform, converse warning, and J-invariant subspaces.
Omega0 = Omega
J0 = J
D = np.diag([1.6, 0.7, 1/1.6, 1/0.7])
Omega1 = D.T @ Omega0 @ D
# Both are compatible with J0 in this chosen diagonal example.
G0 = Omega0 @ J0
G1 = Omega1 @ J0
assert np.min(np.linalg.eigvalsh(G0)) > 0
assert np.min(np.linalg.eigvalsh(G1)) > 0
path_min_eigs = []
for t in homotopy_t:
    O_t = (1 - t) * Omega0 + t * Omega1
    G_t = O_t @ J0
    path_min_eigs.append(float(np.min(np.linalg.eigvalsh(G_t))))
assert min(path_min_eigs) > 0

# Source counterexample warning: a symplectic path from Omega to -Omega cannot share one compatible J at endpoints.
def source_counterexample_omega(t):
    c, s = np.cos(np.pi * t), np.sin(np.pi * t)
    M = np.zeros((4, 4))
    M[0, 2] = c; M[2, 0] = -c
    M[0, 3] = s; M[3, 0] = -s
    M[2, 1] = s; M[1, 2] = -s
    M[1, 3] = c; M[3, 1] = -c
    return M
counter_dets = [float(np.linalg.det(source_counterexample_omega(t))) for t in homotopy_t]
endpoint_positive = float(np.min(np.linalg.eigvalsh(source_counterexample_omega(0) @ J0)))
endpoint_negative = float(np.max(np.linalg.eigvalsh(source_counterexample_omega(1) @ J0)))
assert min(abs(value) for value in counter_dets) > 0.99
assert endpoint_positive > 0
assert endpoint_negative < 0

B_j_invariant = np.eye(4)[:, [0, 2]]  # span{x1,y1}
B_not_invariant = np.eye(4)[:, [0, 1]]  # span{x1,x2}
restricted_good = B_j_invariant.T @ Omega @ B_j_invariant
restricted_bad = B_not_invariant.T @ Omega @ B_not_invariant
restricted_good_det = float(np.linalg.det(restricted_good))
restricted_bad_norm = float(np.linalg.norm(restricted_bad))
J_invariance_good = float(np.linalg.norm((np.eye(4) - B_j_invariant @ np.linalg.pinv(B_j_invariant)) @ J @ B_j_invariant))
J_invariance_bad = float(np.linalg.norm((np.eye(4) - B_not_invariant @ np.linalg.pinv(B_not_invariant)) @ J @ B_not_invariant))
assert abs(restricted_good_det) > 0.99
assert restricted_bad_norm < 1e-12
assert J_invariance_good < 1e-12
assert J_invariance_bad > 0.99

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
axes[0].plot(homotopy_t, path_min_eigs, color="#1b9e77", lw=2)
axes[0].set_title("J-compatible forms: convex path stays symplectic")
axes[0].set_xlabel("t")
axes[0].set_ylabel("min eig(Omega_t J)")
axes[1].plot(homotopy_t, np.abs(counter_dets), color="#2c7fb8", lw=2, label="|det omega_t|")
axes[1].axhline(1, color="0.55", ls="--", lw=1)
axes[1].set_title("converse warning: path stays nondegenerate")
axes[1].set_xlabel("t")
axes[1].set_ylabel("determinant magnitude")
axes[2].bar(["J-invariant\nplane", "not J-invariant\nplane"], [abs(restricted_good_det), restricted_bad_norm], color=["#1b9e77", "#d95f02"])
axes[2].set_yscale("symlog", linthresh=1e-14)
axes[2].set_title("almost-complex submanifold test")
axes[2].set_ylabel("restricted omega witness")
fig.suptitle("Consequences of compatible triples: deformation, warnings, and submanifolds")
consequences_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "compatible-triples-consequences-ledger.png")
plt.close(fig)
display_artifact(consequences_path, width=760)
print({"path_min_eig_floor": min(path_min_eigs), "counter_det_floor": min(abs(value) for value in counter_dets), "restricted_good_det": restricted_good_det})

## Reading The Visuals

The reconstruction diagram is the lecture table turned into checks. From `(omega,J)` we get `g`; from `(g,J)` we get `omega`; from `(omega,g)` the polar-decomposition route gives `J`. The remaining questions are differential rather than purely algebraic: flatness of `g`, closedness of `omega`, and integrability of `J`.

The contractibility model follows the homework idea. Compatible complex structures can be encoded by a Lagrangian graph transverse to a fixed Lagrangian and a positive metric on that fixed Lagrangian. Symmetric graph matrices contract linearly to zero, and positive metrics contract convexly to the identity, so the model makes the contractible choice visible.

The consequences ledger separates two statements that are easy to conflate. If two symplectic forms share a compatible `J`, their convex path remains symplectic because the induced metrics stay positive. But a path of symplectic forms alone does not force a common compatible `J`; the source's `R4` family is represented by a nondegenerate path whose endpoints cannot both tame the same `J`.

In [ ]:
residuals = {
    "metric_reconstruction_residual": metric_reconstruction_residual,
    "omega_reconstruction_residual": omega_reconstruction_residual,
    "J_reconstruction_residual": J_reconstruction_residual,
    "orthogonality_residual": orthogonality_residual,
    "hermitian_symplectic_residual": hermitian_symplectic_residual,
    "contractible_graph_final_norm": S_norms[-1],
    "contractible_metric_min_eig_floor": min(P_min_eigs),
    "compatible_deformation_min_metric_eig": min(path_min_eigs),
    "counterexample_min_abs_det": min(abs(value) for value in counter_dets),
    "counterexample_endpoint_positive_floor": endpoint_positive,
    "counterexample_endpoint_negative_ceiling": endpoint_negative,
    "J_invariant_restricted_omega_det": restricted_good_det,
    "non_J_invariant_restricted_omega_norm": restricted_bad_norm,
}
residual_path = save_json(residuals, ARTIFACT_TOPIC, "checks", "compatible-triples-residuals.json")

source_span = {
    "title": "Compatible Triples",
    "printed_pages": "72-75",
    "physical_pdf_pages": "86-89",
    "source_file": "Lectures on Symplectic Geometry.pdf",
    "sections": [
        "Compatibility",
        "Triple of structures",
        "First consequences",
        "Homework 9 contractibility themes",
    ],
    "concepts": [
        "compatible triple",
        "Hermitian metric",
        "contractible choice",
        "orthogonality",
        "deformation-equivalent symplectic forms",
        "almost complex submanifold",
    ],
    "copyright_note": "Source pages were used for terminology and coverage; notebook prose, visuals, and code are original and no page crops or long exercise text are copied.",
}
source_path = save_json(source_span, ARTIFACT_TOPIC, "checks", "source-span.json")

storyboard = {
    "chapter_goal": "Make compatible triples function as reconstructable algebraic data with contractible choices and concrete consequences.",
    "source_span_read": source_span,
    "library_routing": [
        {"concept": "compatible triple", "representation": "reconstruction graph and residual ledger", "library": "networkx + numpy", "why": "the lecture's table is a directed dependency structure plus matrix identities"},
        {"concept": "Hermitian metric and orthogonality", "representation": "J.T G J and J.T Omega J residuals", "library": "numpy", "why": "orthogonality and Hermitian compatibility are finite matrix equations"},
        {"concept": "contractible choice", "representation": "symmetric graph and positive-metric homotopies", "library": "numpy + matplotlib", "why": "Homework 9's model is a contractible vector/convex space"},
        {"concept": "first consequences", "representation": "metric positivity and subspace restriction plots", "library": "numpy + matplotlib", "why": "deformation and submanifold claims are positivity/nondegeneracy checks"},
    ],
    "visual_sequence": [
        {"concept": "omega-J-g reconstruction diagram", "artifact": "artifacts/lecture-13/figures/omega-J-g-reconstruction-diagram.png", "inspection_target": "each pair reconstructs the third and leaves a differential question"},
        {"concept": "contractible choice", "artifact": "artifacts/lecture-13/figures/contractible-compatible-choice-model.png", "inspection_target": "symmetric Lagrangian graphs and positive metrics retract"},
        {"concept": "compatible triples consequences", "artifact": "artifacts/lecture-13/figures/compatible-triples-consequences-ledger.png", "inspection_target": "shared J gives deformation while non-J-invariant subspaces fail the restricted form test"},
    ],
    "computational_checks": residuals,
}
storyboard_path = save_json(storyboard, ARTIFACT_TOPIC, "checks", "visual-storyboard.json")

final_sanity = {
    "artifacts": [
        str(triple_path.relative_to(BOOK_ROOT)),
        str(contractible_path.relative_to(BOOK_ROOT)),
        str(consequences_path.relative_to(BOOK_ROOT)),
        str(residual_path.relative_to(BOOK_ROOT)),
        str(source_path.relative_to(BOOK_ROOT)),
        str(storyboard_path.relative_to(BOOK_ROOT)),
    ],
    "assertions": {
        "triple_reconstructs_metric": bool(metric_reconstruction_residual < 1e-12),
        "triple_reconstructs_omega": bool(omega_reconstruction_residual < 1e-12),
        "triple_reconstructs_J": bool(J_reconstruction_residual < 1e-12),
        "J_is_g_orthogonal": bool(orthogonality_residual < 1e-12),
        "contractible_model_retracts_graph": bool(S_norms[-1] < 1e-12),
        "metric_homotopy_stays_positive": bool(min(P_min_eigs) > 0),
        "shared_J_deformation_stays_symplectic": bool(min(path_min_eigs) > 0),
        "counterexample_path_stays_nondegenerate": bool(min(abs(value) for value in counter_dets) > 0.99),
        "J_invariant_subspace_is_symplectic": bool(abs(restricted_good_det) > 0.99),
        "non_J_invariant_plane_fails_restriction_test": bool(restricted_bad_norm < 1e-12 and J_invariance_bad > 0.99),
    },
}
final_path = save_json(final_sanity, ARTIFACT_TOPIC, "checks", "final-sanity.json")
print(json.dumps(final_sanity, indent=2))

In [ ]:
final_sanity = read_json(ARTIFACT_ROOT / "checks" / "final-sanity.json")
assert all(final_sanity["assertions"].values())
for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    assert path.exists(), relative
    assert path.stat().st_size > 100, relative

for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    if path.suffix.lower() in {".png", ".html", ".svg"}:
        display_artifact(path, width=760, height=430 if path.suffix == ".html" else None)

print("Lecture 13 checks passed; artifacts are present and nonempty.")

## Takeaways

- A compatible triple lets `omega`, `J`, and `g` reconstruct one another when the positivity and orthogonality hypotheses are present.
- The compatible choices are contractible, not just path-connected; Homework 9 models this by Lagrangian graph data and positive metrics.
- If two symplectic forms are compatible with the same `J`, their convex interpolation remains symplectic because the induced metrics remain positive.
- The converse fails: a deformation path of symplectic forms does not guarantee one shared compatible `J`.
- An almost-complex submanifold of a symplectic manifold with compatible `J` is symplectic because the restricted metric and two-form remain nondegenerate.